In [ ]:
!pip install groq python-dotenv

NAME: KUSUMITA Y
SRN:PES2UG23CS718
SEC:K

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

In [ ]:
from groq import Groq
import os
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a technical support expert. Give precise coding solutions."
    },
    "billing": {
        "system_prompt": "You are a billing support agent. Be polite and explain refund policies."
    },
    "general": {
        "system_prompt": "You are a friendly general assistant."
    }
}

In [ ]:
def route_prompt(user_input):
    prompt = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the word.

Text: {user_input}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()

In [ ]:
def process_request(user_input):

    category = route_prompt(user_input)

    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return category, response.choices[0].message.content

In [ ]:
while True:
    user_input = input("Enter your query (type exit to stop): ")

    if user_input.lower() == "exit":
        break

    category, answer = process_request(user_input)

    print("Routed to:", category)
    print("Response:", answer)
    print("-"*50)